# Forecasting Analysis

This notebook turns the forecasting artifacts produced by `scripts/forecasting_analysis.py` into report-ready tables and figures. It compares cross-sectional predictive quality across deep learning models and an explicit baseline model.

## Setup

The analysis uses only persisted outputs, so it is reproducible and does not retrain any model.

In [1]:
from pathlib import Path
import os

Path("/tmp/matplotlib-cache").mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-cache")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("default")
plt.rcParams.update({
    "figure.figsize": (10, 5),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
elif not (ROOT / "outputs").exists() and (ROOT / "ML_For_Finance_Project-AxelTurinPlessia-362559-ClementMeddeb-346164").exists():
    ROOT = ROOT / "ML_For_Finance_Project-AxelTurinPlessia-362559-ClementMeddeb-346164"

TABLE_DIR = ROOT / "outputs" / "tables"
FIGURE_DIR = ROOT / "outputs" / "figures"
PREDICTION_DIR = ROOT / "outputs" / "predictions"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

summary = pd.read_csv(TABLE_DIR / "forecasting_summary.csv")
monthly_ic = pd.read_csv(TABLE_DIR / "monthly_rank_ic.csv", parse_dates=["MthCalDt"])
monthly_spread = pd.read_csv(TABLE_DIR / "monthly_spread.csv", parse_dates=["MthCalDt"])
predictions = pd.read_parquet(PREDICTION_DIR / "dl_xgb_predictions.parquet")
predictions["MthCalDt"] = pd.to_datetime(predictions["MthCalDt"])

print(f"Loaded {len(summary)} forecasting rows and {len(predictions):,} prediction rows from {ROOT}")

/tmp/matplotlib-cache is not a writable directory
Matplotlib created a temporary cache directory at /tmp/matplotlib-_9p4ncqw because there was an issue with the default path (/tmp/matplotlib-cache); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


FileNotFoundError: [Errno 2] No such file or directory: '/home/aturin/ML/ML_For_Finance_Project-AxelTurinPlessia-362559-ClementMeddeb-346164/outputs/tables/forecasting_summary.csv'

## Baseline Definition

The baseline is selected from the non-deep-learning family using validation Rank IC, then shown explicitly as `baseline_model` in every comparison. The notebook reads `outputs/tables/forecasting_summary.csv` directly, so the report tables and the notebook output use the same source numbers.


In [ ]:
baseline_candidates = [
    "xgb_classifier", "xgb_er_train", "ridge", "elastic_net", "logistic_classifier",
    "gb_classifier", "gradient_boosting_reg", "naive_momentum", "naive_reversal",
]
available_baselines = summary.loc[
    summary["model"].astype(str).isin(baseline_candidates) & summary["split"].eq("validation")
].copy()
if available_baselines.empty:
    raise ValueError("No baseline model found in forecasting_summary.csv")

baseline_source_model = available_baselines.sort_values("mean_rank_ic", ascending=False).iloc[0]["model"]
print("Selected baseline source model:", baseline_source_model)

MODEL_LABELS = {
    "mlp_classifier": "MLP",
    "ft_classifier": "FT",
    "temporal_classifier": "Temporal",
    baseline_source_model: "baseline_model",
}

SCORE_COLUMN_BY_MODEL = {
    "mlp_classifier": "prediction_mlp_classifier_score",
    "ft_classifier": "prediction_ft_classifier_score",
    "temporal_classifier": "prediction_temporal_classifier_score",
    "xgb_classifier": "prediction_xgb_classifier_score",
    "xgb_er_train": "prediction_xgb_regressor",
}

def label_model(model):
    return MODEL_LABELS.get(str(model), str(model))

def include_baseline_top_n(data, metric, split=None, n=5, ascending=False):
    work = data.copy()
    if split is not None:
        work = work.loc[work["split"].eq(split)].copy()
    ranked = work.sort_values(metric, ascending=ascending).head(n)
    baseline = work.loc[work["model"].eq(baseline_source_model)]
    out = pd.concat([ranked, baseline], ignore_index=True).drop_duplicates(subset=["model", "split"])
    out = out.sort_values(metric, ascending=ascending).copy()
    out["model_label"] = out["model"].map(label_model)
    return out

summary_report = summary.copy()
summary_report["model_label"] = summary_report["model"].map(label_model)
summary_report.to_csv(TABLE_DIR / "forecasting_summary_report_ready.csv", index=False)


## 1. Model Ranking Tables

The tables rank models by validation Rank IC, test Rank IC, hit rate, and top-bottom spread. The baseline row is included even when it is outside the top five.

In [ ]:
ranking_specs = [
    ("validation_rank_ic", "validation", "mean_rank_ic", False),
    ("test_rank_ic", "test", "mean_rank_ic", False),
    ("test_hit_rate", "test", "hit_rate", False),
    ("test_spread", "test", "mean_top_bottom_spread", False),
]
ranking_tables = {}
for name, split, metric, ascending in ranking_specs:
    table = include_baseline_top_n(summary, metric=metric, split=split, n=5, ascending=ascending)
    cols = ["model_label", "model", "split", "mean_rank_ic", "rank_ic_tstat", "hit_rate", "mean_top_bottom_spread", "spread_tstat", "num_months"]
    table = table[cols]
    table.to_csv(TABLE_DIR / f"forecasting_{name}_top5_plus_baseline.csv", index=False)
    ranking_tables[name] = table
    print(f"\n{name}")
    print(table.to_string(index=False))

## 2. Bar Plots

The plots compare the main deep-learning classifiers with the explicit baseline on the test sample. Rank IC measures cross-sectional ordering, spread measures economic separation between top and bottom predicted buckets, and hit rate measures monthly consistency.

In [ ]:
plot_models = ["ft_classifier", "mlp_classifier", "temporal_classifier", baseline_source_model]
plot_df = summary.loc[summary["split"].eq("test") & summary["model"].isin(plot_models)].copy()
plot_df["model_label"] = plot_df["model"].map(label_model)
plot_df = plot_df.drop_duplicates("model_label")

plot_specs = [
    ("mean_rank_ic", "Mean Rank IC", "forecasting_rank_ic_comparison.png"),
    ("mean_top_bottom_spread", "Mean top-bottom spread", "forecasting_spread_comparison.png"),
    ("hit_rate", "Hit rate", "forecasting_hit_rate_comparison.png"),
]
for metric, title, filename in plot_specs:
    ordered = plot_df.sort_values(metric, ascending=True)
    fig, ax = plt.subplots(figsize=(8, 4.5))
    colors = ["#8C564B" if label == "baseline_model" else "#4C78A8" for label in ordered["model_label"]]
    ax.barh(ordered["model_label"], ordered[metric], color=colors)
    ax.set_title(title)
    ax.set_xlabel(metric)
    for y, value in enumerate(ordered[metric]):
        ax.text(value, y, f" {value:.3f}", va="center")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / filename, dpi=160, bbox_inches="tight")
    plt.show()

## 3. Score Correlation Heatmap

The heatmap reports Spearman correlations between model scores on the validation split. The baseline column uses the same baseline selected above, so the correlation diagnostic is consistent with the ranking tables. Lower correlations suggest complementary rankings; high correlations suggest similar information sets.


In [ ]:
score_columns = {
    "MLP": SCORE_COLUMN_BY_MODEL["mlp_classifier"],
    "FT": SCORE_COLUMN_BY_MODEL["ft_classifier"],
    "Temporal": SCORE_COLUMN_BY_MODEL["temporal_classifier"],
    "baseline_model": SCORE_COLUMN_BY_MODEL.get(baseline_source_model),
}
if score_columns["baseline_model"] is None:
    raise ValueError(f"No prediction score column mapping for baseline model {baseline_source_model}")

missing = [col for col in score_columns.values() if col not in predictions.columns]
if missing:
    raise ValueError(f"Missing score columns: {missing}")

corr_data = predictions.loc[predictions["split"].eq("validation"), list(score_columns.values())].rename(
    columns={v: k for k, v in score_columns.items()}
)
score_corr = corr_data.corr(method="spearman")
score_corr.to_csv(TABLE_DIR / "forecasting_validation_score_spearman_correlation_core_models.csv")

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(score_corr.to_numpy(), vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(np.arange(len(score_corr.columns)))
ax.set_yticks(np.arange(len(score_corr.index)))
ax.set_xticklabels(score_corr.columns, rotation=35, ha="right")
ax.set_yticklabels(score_corr.index)
for i in range(score_corr.shape[0]):
    for j in range(score_corr.shape[1]):
        ax.text(j, i, f"{score_corr.iloc[i, j]:.2f}", ha="center", va="center", color="black")
ax.set_title("Validation score Spearman correlation")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "forecasting_core_score_correlation_heatmap.png", dpi=160, bbox_inches="tight")
plt.show()
score_corr


## 4. Stability Analysis

A stable model should retain similar Rank IC when moving from validation to test. The table reports validation IC, test IC, and the test-minus-validation difference.

In [ ]:
stability_models = ["ft_classifier", "mlp_classifier", "temporal_classifier", baseline_source_model]
ic_stability = summary.loc[summary["model"].isin(stability_models)].pivot(index="model", columns="split", values="mean_rank_ic")
ic_stability = ic_stability.rename(columns={"validation": "IC_validation", "test": "IC_test"})
ic_stability["difference"] = ic_stability["IC_test"] - ic_stability["IC_validation"]
ic_stability = ic_stability.reset_index()
ic_stability["model_label"] = ic_stability["model"].map(label_model)
ic_stability = ic_stability[["model_label", "model", "IC_validation", "IC_test", "difference"]].sort_values("IC_test", ascending=False)
ic_stability.to_csv(TABLE_DIR / "forecasting_ic_stability_core_models.csv", index=False)
print(ic_stability.to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 5))
colors = ["#8C564B" if label == "baseline_model" else "#4C78A8" for label in ic_stability["model_label"]]
ax.scatter(ic_stability["IC_validation"], ic_stability["IC_test"], s=90, c=colors)
lo = min(ic_stability["IC_validation"].min(), ic_stability["IC_test"].min()) * 0.95
hi = max(ic_stability["IC_validation"].max(), ic_stability["IC_test"].max()) * 1.05
ax.plot([lo, hi], [lo, hi], color="black", linewidth=1, linestyle="--")
for _, row in ic_stability.iterrows():
    ax.annotate(row["model_label"], (row["IC_validation"], row["IC_test"]), xytext=(5, 5), textcoords="offset points")
ax.set_xlabel("Validation mean Rank IC")
ax.set_ylabel("Test mean Rank IC")
ax.set_title("Validation vs test Rank IC")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "forecasting_validation_vs_test_ic.png", dpi=160, bbox_inches="tight")
plt.show()

## 5. Spread Dynamics

Cumulative top-bottom spread translates forecasts into a simple long-short diagnostic: each month is the realized return gap between stocks ranked high and low by the model.

In [ ]:
spread_models = ["ft_classifier", "mlp_classifier", "temporal_classifier", baseline_source_model]
spread_dynamics = monthly_spread.loc[
    monthly_spread["split"].eq("test") & monthly_spread["model"].isin(spread_models)
].copy()
spread_dynamics["model_label"] = spread_dynamics["model"].map(label_model)
spread_dynamics = spread_dynamics.sort_values(["model_label", "MthCalDt"])
spread_dynamics["cumulative_spread"] = spread_dynamics.groupby("model_label")["spread_t"].cumsum()
spread_dynamics.to_csv(TABLE_DIR / "forecasting_test_cumulative_spread_core_models.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 5))
for label, part in spread_dynamics.groupby("model_label"):
    color = "#8C564B" if label == "baseline_model" else None
    ax.plot(part["MthCalDt"], part["cumulative_spread"], label=label, linewidth=2, color=color)
ax.axhline(0, color="black", linewidth=1)
ax.set_title("Test cumulative top-bottom spread")
ax.set_xlabel("Month")
ax.set_ylabel("Cumulative spread")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "forecasting_cumulative_spread_core_models.png", dpi=160, bbox_inches="tight")
plt.show()

## Interpretation

Rank IC is the primary forecasting metric because the task is cross-sectional stock ranking. Hit rate reports how often the monthly Rank IC is positive, and top-bottom spread translates the ranking into a simple economic long-short diagnostic.

The temporal classifier is compared against MLP and FT through both performance and score correlation. When its correlation with the tabular models is materially below one, the temporal signal is complementary rather than a simple rescaling of the same cross-section. When the correlations are high, its incremental portfolio value should be judged more cautiously.

The baseline gap is economically meaningful but not absolute. The XGBoost baseline is selected by validation Rank IC and kept as `baseline_model`, so deep learning is judged against a strong non-DL benchmark using the same summary table used in the report. Stability is assessed by the validation-to-test IC difference: small differences indicate robust generalization, while a large negative move points to validation-specific overfitting.


## Output Manifest

In [ ]:
manifest = pd.DataFrame([
    {"artifact": "forecasting_summary_report_ready", "path": TABLE_DIR / "forecasting_summary_report_ready.csv"},
    {"artifact": "validation_rank_ic_top5_plus_baseline", "path": TABLE_DIR / "forecasting_validation_rank_ic_top5_plus_baseline.csv"},
    {"artifact": "test_rank_ic_top5_plus_baseline", "path": TABLE_DIR / "forecasting_test_rank_ic_top5_plus_baseline.csv"},
    {"artifact": "test_hit_rate_top5_plus_baseline", "path": TABLE_DIR / "forecasting_test_hit_rate_top5_plus_baseline.csv"},
    {"artifact": "test_spread_top5_plus_baseline", "path": TABLE_DIR / "forecasting_test_spread_top5_plus_baseline.csv"},
    {"artifact": "core_score_correlation", "path": TABLE_DIR / "forecasting_validation_score_spearman_correlation_core_models.csv"},
    {"artifact": "ic_stability", "path": TABLE_DIR / "forecasting_ic_stability_core_models.csv"},
    {"artifact": "cumulative_spread", "path": TABLE_DIR / "forecasting_test_cumulative_spread_core_models.csv"},
    {"artifact": "rank_ic_plot", "path": FIGURE_DIR / "forecasting_rank_ic_comparison.png"},
    {"artifact": "spread_plot", "path": FIGURE_DIR / "forecasting_spread_comparison.png"},
    {"artifact": "hit_rate_plot", "path": FIGURE_DIR / "forecasting_hit_rate_comparison.png"},
    {"artifact": "correlation_heatmap", "path": FIGURE_DIR / "forecasting_core_score_correlation_heatmap.png"},
    {"artifact": "stability_plot", "path": FIGURE_DIR / "forecasting_validation_vs_test_ic.png"},
    {"artifact": "spread_dynamics_plot", "path": FIGURE_DIR / "forecasting_cumulative_spread_core_models.png"},
])
manifest["path"] = manifest["path"].astype(str)
manifest.to_csv(TABLE_DIR / "forecasting_analysis_notebook14_manifest.csv", index=False)
manifest
